In [0]:
#Silver Layer

# Import all built-in PySpark SQL functions
# Examples: col(), year(), month(), to_timestamp(), sum(), avg(), etc.
from pyspark.sql.functions import *

bronze_df = spark.read \
    .format("delta") \
    .load("/Volumes/workspace/default/e-commerce-data/bronze")

# Create the Silver DataFrame from the Bronze DataFrame
silver_df = (

    # Start with the Bronze DataFrame
    bronze_df

        # -------------------------------------------------------------
        # Remove duplicate rows
        # -------------------------------------------------------------
        # If two or more rows are exactly identical,
        # only one copy is kept.
        #
        # Example:
        # InvoiceNo  StockCode  Quantity
        # 10001      A123       10
        # 10001      A123       10
        #
        # After dropDuplicates():
        #
        # 10001      A123       10
        #
        .dropDuplicates()

        # -------------------------------------------------------------
        # Keep only rows where Quantity is greater than zero
        # -------------------------------------------------------------
        # Negative quantities usually represent returned items.
        # For sales analysis, we only keep actual purchases.
        #
        # Example:
        #
        # Quantity = 5    -> Keep
        # Quantity = -2   -> Remove
        #
        .filter(col("Quantity") > 0)

        # -------------------------------------------------------------
        # Keep only rows where UnitPrice is greater than zero
        # -------------------------------------------------------------
        # Removes invalid or free items that may have been
        # entered incorrectly.
        #
        .filter(col("UnitPrice") > 0)

        # -------------------------------------------------------------
        # Remove rows where CustomerID is NULL
        # -------------------------------------------------------------
        # Some transactions do not have a customer ID.
        # Since customer analysis is required later,
        # these records are removed.
        #
        .filter(col("CustomerID").isNotNull())

        # -------------------------------------------------------------
        # Convert InvoiceDate from String to Timestamp
        # -------------------------------------------------------------
        # Bronze:
        # "12/1/2010 8:26"
        #
        # Silver:
        # 2010-12-01 08:26:00
        #
        # Timestamp allows us to perform date calculations.
        #
        .withColumn(
            "InvoiceDate",
            to_timestamp("InvoiceDate", "M/d/yyyy H:mm")
        )

        # -------------------------------------------------------------
        # Calculate TotalAmount
        # -------------------------------------------------------------
        # Formula:
        #
        # TotalAmount = Quantity × UnitPrice
        #
        # Example:
        #
        # Quantity = 10
        # UnitPrice = 2.55
        #
        # TotalAmount = 25.50
        #
        .withColumn(
            "TotalAmount",
            col("Quantity") * col("UnitPrice")
        )
)

silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .save("/Volumes/workspace/default/e-commerce-data/silver")